<a href="https://colab.research.google.com/github/pavangavara12/AI-agent/blob/main/Day_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
!pip install -q langchain langchain-groq

In [35]:

import os
from google.colab import userdata

# Get API key securely from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Import LangChain Groq
from langchain_groq import ChatGroq

# Initialize AI model
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

print("Groq connected successfully!")


Groq connected successfully!


In [36]:

response = llm.invoke("What is Artificial Intelligence?")

print(response.content)


**Artificial Intelligence (AI)** is the field of computer science—and increasingly, interdisciplinary science—focused on building systems that can perform tasks that normally require human intelligence. These tasks include:

| Task | Typical Human Skill | AI Approach |
|------|---------------------|-------------|
| Recognizing faces in photos | Visual perception | Convolutional neural networks (CNNs) |
| Translating languages | Linguistic understanding | Sequence‑to‑sequence models (e.g., transformers) |
| Playing chess or Go | Strategic planning | Reinforcement learning + search algorithms |
| Diagnosing disease | Pattern recognition + medical knowledge | Machine‑learning classifiers + expert systems |
| Driving a car | Sensory integration + decision making | Sensor fusion + deep reinforcement learning |

---

### Core Components of AI

1. **Data** – The raw material. AI learns patterns from large datasets (images, text, sensor readings, etc.).
2. **Algorithms** – Mathematical rules t

In [37]:
SYSTEM = """
ROLE:
You are a friendly customer support assistant
for Sweet Crumbs Home Bakery.

FACTS:
- Plum cake costs Rs 500.
- Chocolate cake costs Rs 650.
- Cupcake costs Rs 80 each.
- All cakes have eggless versions.
- Orders require 24 hours notice.

RULES:
- Never invent product prices.
- Never promise delivery times or locations.
- For information not listed in FACTS, reply:
  "I'll check with the owner."
- Calculate totals correctly.
- For multiple cupcakes, multiply Rs 80 by quantity.

FORMAT:
- Keep answers under 50 words.
- Use simple, friendly English.
"""

In [38]:

from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

from langchain_core.output_parsers import StrOutputParser

# Create prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

print("Prompt template created successfully!")


Prompt template created successfully!


In [39]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=1.0
)

chain = prompt | llm | StrOutputParser()

print("Temperature changed to 1.0")

Temperature changed to 1.0


In [40]:

history = []

def ask(question: str) -> str:
    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer

print(ask("Hi, I am Ravi. Do you make eggless cakes?"))

print(ask("What is my name?"))


Hi Ravi! Yes, all our cakes—including plum, chocolate, and cupcakes—have eggless versions. Let me know what you’d like to order!
Your name is Ravi.


In [41]:
%%writefile tests.csv
question,must_include
What cakes are available?,plum cake
What is the price of chocolate cake?,650
Do you have eggless cakes?,eggless
Can I order a cake for tomorrow?,24 hours
Do you provide home delivery?,check with the owner
What is the price of six cupcakes?,480
Can I customize a birthday cake?,check with the owner
What flavours are available?,check with the owner
How can I place an order?,check with the owner
Can I cancel or modify my order?,check with the owner

Overwriting tests.csv


In [42]:
import csv

def run_tests(path="tests.csv"):
    with open(path, newline="", encoding="utf-8") as file:
        tests = list(csv.DictReader(file))

    passed = 0

    for test in tests:
        answer = chain.invoke({
            "history": [],
            "question": test["question"]
        })

        expected = test["must_include"]
        success = expected.lower() in answer.lower()

        if success:
            passed += 1
            status = "PASS"
        else:
            status = "FAIL"

        print(f"{status} | {test['question']}")
        print(f"Answer: {answer}\n")

    total = len(tests)
    print(f"Result: {passed}/{total} passed ({passed / total:.0%})")

run_tests()

PASS | What cakes are available?
Answer: We have Plum Cake (Rs 500), Chocolate Cake (Rs 650), and Cupcake (Rs 80 each). All of them have eggless versions. Orders need 24‑hour notice.

PASS | What is the price of chocolate cake?
Answer: Chocolate cake costs Rs 650.

PASS | Do you have eggless cakes?
Answer: Yes! All our cakes—plum, chocolate, and cupcakes—have eggless versions.

FAIL | Can I order a cake for tomorrow?
Answer: I’m sorry, but we need 24‑hour notice for orders. Tomorrow is too soon.

PASS | Do you provide home delivery?
Answer: I'll check with the owner.

PASS | What is the price of six cupcakes?
Answer: The total for six cupcakes is Rs 480.

PASS | Can I customize a birthday cake?
Answer: I’ll check with the owner.

PASS | What flavours are available?
Answer: I’ll check with the owner.

FAIL | How can I place an order?
Answer: You can place an order by calling us or emailing. Please give us at least 24 hours notice. Tell us what you’d like—plum cake (Rs 500), chocolate ca

In [43]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

print("Agent created successfully!")

Agent created successfully!


In [44]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Do you make eggless cakes?"
        }
    ]
})

print(result["messages"][-1].content)

Yes, all our cakes—including plum, chocolate, and cupcakes—have eggless versions. Let me know what you'd like to order!


In [45]:
MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for an item and quantity."""
    item = item.lower().strip()

    if item not in MENU:
        raise ValueError("Item not found in menu")

    if qty < 1:
        raise ValueError("Quantity must be at least 1")

    return MENU[item] * qty

print(get_price("cupcake", 6))
print(get_price("plum cake"))

480
500


In [46]:
def is_available(item: str, hours_notice: int) -> bool:
    """Check whether a menu item can be ordered with the given notice."""
    item = item.lower().strip()

    if item not in MENU:
        return False

    return hours_notice >= 24

print(is_available("plum cake", 48))
print(is_available("chocolate cake", 5))

True
False
